In [1]:
import pandas as pd
icd10_map_dx = pd.read_csv("CCS/DXCCSR_v2026-1.csv", dtype=str).rename(columns=lambda x: x.strip("'")).applymap(lambda x: x.strip("'").strip() if isinstance(x, str) else x)
icd10_map_pr = pd.read_csv("CCS/PRCCSR_v2026-1.csv", dtype=str).rename(columns=lambda x: x.strip("'")).applymap(lambda x: x.strip("'").strip() if isinstance(x, str) else x)
icd9_map_dx = pd.read_csv("CCS/ccs_multi_dx_tool_2015.csv", dtype=str).rename(columns=lambda x: x.strip("'")).applymap(lambda x: x.strip("'").strip() if isinstance(x, str) else x)
icd9_map_pr = pd.read_csv("CCS/ccs_multi_pr_tool_2015.csv", dtype=str).rename(columns=lambda x: x.strip("'")).applymap(lambda x: x.strip("'").strip() if isinstance(x, str) else x)

/tmp/ipykernel_191678/592224789.py:2: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  icd10_map_dx = pd.read_csv("CCS/DXCCSR_v2026-1.csv", dtype=str).rename(columns=lambda x: x.strip("'")).applymap(lambda x: x.strip("'").strip() if isinstance(x, str) else x)
/tmp/ipykernel_191678/592224789.py:3: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  icd10_map_pr = pd.read_csv("CCS/PRCCSR_v2026-1.csv", dtype=str).rename(columns=lambda x: x.strip("'")).applymap(lambda x: x.strip("'").strip() if isinstance(x, str) else x)
/tmp/ipykernel_191678/592224789.py:4: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  icd9_map_dx = pd.read_csv("CCS/ccs_multi_dx_tool_2015.csv", dtype=str).rename(columns=lambda x: x.strip("'")).applymap(lambda x: x.strip("'").strip() if isinstance(x, str) else x)
/tmp/ipykernel_191678/592224789.py:5: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.m

In [5]:
icd10_map_dx[icd10_map_dx['ICD-10-CM CODE'].str.startswith("A09".replace(".",""))][['ICD-10-CM CODE', 'ICD-10-CM CODE DESCRIPTION', "CCSR CATEGORY 1 DESCRIPTION"]].values
icd9_map_pr.columns

Index(['ICD-9-CM CODE', 'CCS LVL 1', 'CCS LVL 1 LABEL', 'CCS LVL 2',
       'CCS LVL 2 LABEL', 'CCS LVL 3', 'CCS LVL 3 LABEL'],
      dtype='object')

In [ ]:
icd10_map_pr[icd10_map_pr['ICD-10-PCS'].str.startswith("O")][['ICD-10-PCS', 'ICD-10-PCS DESCRIPTION', "PRCCSR DESCRIPTION"]].values

In [ ]:
icd10_map_dx.columns

In [ ]:
df = pd.read_csv("files/icd10_infections.csv")
df['ccs'] = df['ICD-10-CM'].apply(lambda x: icd10_map_dx[icd10_map_dx['ICD-10-CM CODE'].str.startswith(x.replace(".",""))]["CCSR CATEGORY 1 DESCRIPTION"].values[0])
df.to_csv("files/icd10_infections_ccs.csv", index=False)

In [ ]:
df = pd.read_csv("files/icd10pcs_therapies.csv")
df['ccs'] = df['ICD10_PCS'].apply(lambda x: icd10_map_pr[icd10_map_pr['ICD-10-PCS'].str.startswith(x.replace(".",""))]["PRCCSR DESCRIPTION"].values[0] 
                      if len(icd10_map_pr[icd10_map_pr['ICD-10-PCS'].str.startswith(x.replace(".",""))]["PRCCSR DESCRIPTION"].values)>0 else "UNKN")
df.to_csv("files/icd10pcs_therapies_ccs.csv", index=False)

In [ ]:
import json
from tqdm.notebook import tqdm
with open("dataset_14_04_26_EN_newicd.json", "r") as jsonfile: 
    data = json.load(jsonfile)
for p in tqdm(data):
    for f,v in p.items():
        if f == "events":
            for e in v:
                if e['code_type'] == "ICD-10-CM":
                    try:
                        e["ccs"] = icd10_map_dx[icd10_map_dx["ICD-10-CM CODE"] == e['code'].replace(".","")]["CCSR CATEGORY 1 DESCRIPTION"].values[0]
                    except:
                        e["ccs"] = "UNKN"
                elif e['code_type'] == "ICD-10-PCS":
                    try:
                        e["ccs"] = icd10_map_pr[icd10_map_pr["ICD-10-PCS"] == e['code'].replace(".","")]["PRCCSR DESCRIPTION"].values[0]
                    except:
                        e["ccs"] = "UNKN"
                elif e['code_type'] == "CPT":
                    e["ccs"] = e["code_descr"]
                else:
                        e["ccs"] = "UNKN"

with open("data/dataset_14_04_26_EN_newicd_ccs.json", "w") as jsonfile: 
    json.dump(data, jsonfile, indent=4)

In [5]:
import json
from tqdm.notebook import tqdm
with open("mimic-iv_asplenic_1431_new.json", "r") as jsonfile: 
    data = json.load(jsonfile)
for p in tqdm(data):
    for f,v in p.items():
        if f == "events":
            for e in v:
                if e["type"] in ["disease", "infection"] and e['code_type'] == "ICD-10-CM":
                    #try:
                        e["ccs"] = icd10_map_dx[icd10_map_dx["ICD-10-CM CODE"] == e['code']]["CCSR CATEGORY 1 DESCRIPTION"].values[0]
                    #except:
                        #e["ccs"] = "UNKN"
                elif e["type"] in ["therapy", "surgery", "splenectomy"] and e['code_type'] == "ICD-10-PCS":
                    #try:
                        e["ccs"] = icd10_map_pr[icd10_map_pr["ICD-10-PCS"] == e['code']]["PRCCSR DESCRIPTION"].values[0]
                    #except:
                        #e["ccs"] = "UNKN"
                elif e["type"] in ["disease", "infection"] and e['code_type'] == "ICD-9-CM":
                    #try:
                        e["ccs"] = icd9_map_dx[icd9_map_dx["ICD-9-CM CODE"] == e['code']]["CCS LVL 1 LABEL"].values[0]
                    #except:
                        #e["ccs"] = "UNKN"
                elif e["type"] in ["therapy", "surgery", "splenectomy"] and e['code_type'] == "ICD-9-PCS":
                    #try:
                        e["ccs"] = icd9_map_pr[icd9_map_pr["ICD-9-CM CODE"] == e['code']]["CCS LVL 1 LABEL"].values[0]
                    #except:
                        #e["ccs"] = "UNKN"
                elif e['code_type'] == "EMAR-MED":
                        e["ccs"] = e['event']
                else:
                    raise Exception(f"Wrog code type! {e['type']} {e['code_type']}")

  0%|          | 0/1431 [00:00<?, ?it/s]

In [6]:
with open("mimic-iv_asplenic_1431_ccs.json", "w") as jsonfile: 
    json.dump(data, jsonfile, indent=4)

In [7]:
icd9_map_pr.columns

Index(['ICD-9-CM CODE', 'CCS LVL 1', 'CCS LVL 1 LABEL', 'CCS LVL 2',
       'CCS LVL 2 LABEL', 'CCS LVL 3', 'CCS LVL 3 LABEL'],
      dtype='object')

In [84]:
len(set(icd10_map_dx["ICD-10-CM CODE"]).intersection(icds10_dx)) + len(set(icd10_map_pr["ICD-10-PCS"]).intersection(icds10_dx))

4716

In [86]:
len(set(icd9_map_dx["ICD-9-CM CODE"]).intersection(icds9_dx)) + len(set(icd9_map_pr["ICD-9-CM CODE"]).intersection(icds9_dx))

3944

In [91]:
def find_prefix_matches(code, ccs_codes):
    return [c for c in ccs_codes if c.startswith(code)]
def is_prefix(code, ccs_codes):
    return any(c.startswith(code) for c in ccs_codes)
find_prefix_matches('4476',icd9_map_dx["ICD-9-CM CODE"].values)

['4476']

In [89]:
(icds9_dx - set(icd9_map_dx["ICD-9-CM CODE"])) - set(icd9_map_pr["ICD-9-CM CODE"])

set()

In [88]:
(icds10_dx - set(icd10_map_dx["ICD-10-CM CODE"]))- set(icd10_map_pr["ICD-10-PCS"])

set()

In [90]:
len(icdspc - set(icd10_map_pr["ICD-10-PCS"])), len(set(icd9_map_pr["ICD-9-CM CODE"]).intersection(icdspc))

(2127, 0)